# PyMC-20 : MDPs, Bandits et POMDPs

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jsboige/CoursIA/blob/main/MyIA.AI.Notebooks/Probas/_python_port/PyMC-20-Decision-Sequential.ipynb)

## Objectifs d'apprentissage

A la fin de ce notebook, vous saurez :
- Definir un **Processus de Decision Markovien (MDP)** formellement
- Implementer **Value Iteration** et **Policy Iteration**
- Utiliser **RTDP** pour les grands espaces d'etats
- Appliquer le **Reward Shaping** pour accelerer l'apprentissage
- Resoudre le probleme des **Bandits Multi-Bras** (epsilon-greedy, UCB1)
- Modeliser l'incertitude partielle avec les **POMDPs**

**Prerequis** : PyMC-19 (systemes experts), bases de probabilites

**Duree estimee** : 60 minutes

---

| Notebook precedent | Notebook suivant |
|--------------------|------------------|
| [PyMC-19 - Systemes experts](PyMC-19-Decision-Expert-Systems.ipynb) | *Dernier de la serie* |

## 1. Decisions Sequentielles vs One-Shot

### Difference fondamentale

| Type | Caracteristique | Exemple |
|------|-----------------|--------|
| **One-shot** | Decision unique, consequences imediates | Choisir un traitement medical |
| **Sequentielle** | Sequence de decisions, etat change | Piloter un robot, gerer un portefeuille |

Les decisions sequentielles necessitent de **planifier** : anticiper les consequences futures des actions presentes.

Le cadre formel : les **Processus de Decision Markoviens (MDP)**.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
from typing import Dict, List, Tuple, Optional

np.random.seed(42)
print("Environnement pret : numpy + matplotlib")

Environnement pret : numpy + matplotlib


### Environnement pret

Ce notebook utilise uniquement **numpy** et **matplotlib**. Les algorithmes de resolution de MDPs sont implementes from scratch pour comprendre les mecanismes internes.

## 2. Processus de Decision Markoviens (MDP)

### Definition formelle

Un MDP est un tuple $(S, A, P, R, \gamma)$ :

| Element | Notation | Description |
|---------|----------|-------------|
| Etats | $S$ | Ensemble fini d'etats |
| Actions | $A$ | Actions possibles dans chaque etat |
| Transitions | $P(s'|s,a)$ | Probabilite d'atteindre $s'$ depuis $s$ via $a$ |
| Recompenses | $R(s,a,s')$ | Recompense pour la transition |
| Facteur d'escompte | $\gamma \in [0,1)$ | Pondere les recompenses futures |

In [2]:
class GridMDP:
    """MDP sur une grille 2D (Russell & Norvig, 4x3).
    
    Transitions stochastiques : 80% direction voulue, 10% perpendiculaires.
    """
    
    ACTIONS = ['N', 'S', 'E', 'W']
    DIRECTIONS = {'N': (0, 1), 'S': (0, -1), 'E': (1, 0), 'W': (-1, 0)}
    
    def __init__(self, width: int, height: int, gamma: float = 0.9):
        self.width = width
        self.height = height
        self.gamma = gamma
        self.rewards: Dict[Tuple[int,int], float] = {}
        self.terminal_states: set = set()
        self.walls: set = set()
    
    def get_states(self) -> List[Tuple[int,int]]:
        return [(x, y) for x in range(self.width) for y in range(self.height)
                if (x, y) not in self.walls]
    
    def get_reward(self, state: Tuple[int,int]) -> float:
        return self.rewards.get(state, -0.04)
    
    def _next_state(self, state: Tuple[int,int], action: str) -> Tuple[int,int]:
        x, y = state
        dx, dy = self.DIRECTIONS[action]
        nx, ny = x + dx, y + dy
        if nx < 0 or nx >= self.width or ny < 0 or ny >= self.height or (nx, ny) in self.walls:
            return state
        return (nx, ny)
    
    def get_transitions(self, state: Tuple[int,int], action: str) -> List[Tuple[Tuple[int,int], float]]:
        if state in self.terminal_states:
            return [(state, 1.0)]
        
        transitions = [(self._next_state(state, action), 0.8)]
        perp = ['E', 'W'] if action in ('N', 'S') else ['N', 'S']
        for pa in perp:
            transitions.append((self._next_state(state, pa), 0.1))
        return transitions


# Grille classique de Russell & Norvig
mdp = GridMDP(4, 3, gamma=0.9)
mdp.rewards[(3, 2)] = 1.0
mdp.rewards[(3, 1)] = -1.0
mdp.terminal_states = {(3, 2), (3, 1)}
mdp.walls = {(1, 1)}

print("MDP Grille 4x3 cree :")
print("  But (+1) en (3,2)")
print("  Piege (-1) en (3,1)")
print("  Mur en (1,1)")
print(f"  Gamma = {mdp.gamma}")

MDP Grille 4x3 cree :
  But (+1) en (3,2)
  Piege (-1) en (3,1)
  Mur en (1,1)
  Gamma = 0.9


### Visualisation : Propagation des Valeurs dans Value Iteration

Value Iteration propage les valeurs depuis les etats terminaux vers les etats initiaux. A chaque iteration, les valeurs convergent vers $V^*$.

Dans la grille Russell & Norvig :
- Les valeurs les plus elevees sont proches du but (+1)
- Les valeurs sont basses pres du piege (-1)
- Le mur bloque la propagation

## 3. Equation de Bellman

### Intuition

L'equation de Bellman exprime un principe de **consistance temporelle** :

> La valeur d'un etat = recompense immediate + valeur esperee du meilleur etat futur

$$V^*(s) = \max_a \left[ R(s,a) + \gamma \sum_{s'} P(s'|s,a) \cdot V^*(s') \right]$$

Cette equation est le fondement de toutes les methodes de resolution de MDPs.

## 4. Iteration de Valeur

### Algorithme

1. Initialiser $V(s) = 0$ pour tout $s$
2. Repeter jusqu'a convergence :
   - Pour chaque etat $s$ : $V(s) \leftarrow \max_a Q(s,a)$
3. Extraire la politique optimale

La fonction Q-value est :
$$Q(s,a) = R(s,a) + \gamma \sum_{s'} P(s'|s,a) \cdot V(s')$$

In [3]:
def value_iteration(mdp: GridMDP, epsilon: float = 0.001, max_iter: int = 100):
    """Resolution d'un MDP par iteration de valeur."""
    states = mdp.get_states()
    V = {s: 0.0 for s in states}
    
    for iteration in range(max_iter):
        delta = 0
        new_V = dict(V)
        
        for s in states:
            if s in mdp.terminal_states:
                new_V[s] = mdp.get_reward(s)
                continue
            
            q_values = []
            for a in mdp.ACTIONS:
                q = mdp.get_reward(s)
                for next_s, prob in mdp.get_transitions(s, a):
                    q += mdp.gamma * prob * V[next_s]
                q_values.append(q)
            new_V[s] = max(q_values)
            delta = max(delta, abs(V[s] - new_V[s]))
        
        V = new_V
        if delta < epsilon:
            print(f"Convergence apres {iteration + 1} iterations (delta = {delta:.2e})")
            break
    
    # Extraire la politique
    policy = {}
    for s in states:
        if s in mdp.terminal_states:
            policy[s] = 'T'
            continue
        q_values = {}
        for a in mdp.ACTIONS:
            q = mdp.get_reward(s)
            for next_s, prob in mdp.get_transitions(s, a):
                q += mdp.gamma * prob * V[next_s]
            q_values[a] = q
        policy[s] = max(q_values, key=q_values.get)
    
    return V, policy


V_vi, policy_vi = value_iteration(mdp)

arrows = {'N': '^', 'S': 'v', 'E': '>', 'W': '<', 'T': '*'}
print("\nFonction de Valeur V* :")
for y in range(mdp.height - 1, -1, -1):
    row = ""
    for x in range(mdp.width):
        if (x, y) in mdp.walls:
            row += "  ####  "
        else:
            row += f" {V_vi[(x,y)]:6.3f} "
    print(row)

print("\nPolitique optimale :")
for y in range(mdp.height - 1, -1, -1):
    row = ""
    for x in range(mdp.width):
        if (x, y) in mdp.walls:
            row += " # "
        else:
            row += f" {arrows[policy_vi[(x,y)]]} "
    print(row)

Convergence apres 14 iterations (delta = 6.01e-04)

Fonction de Valeur V* :
  0.509   0.650   0.795   1.000 
  0.398   ####    0.486  -1.000 
  0.296   0.254   0.345   0.130 

Politique optimale :
 >  >  >  * 
 ^  #  ^  * 
 ^  >  ^  < 


### Interpretation de l'Iteration de Valeur

**Fonction de valeur V* :**
- Les valeurs decroissent en s'eloignant du but (+1 en (3,2))
- La case (3,1) a une valeur de -1.000 (etat terminal piege)
- Le mur en (1,1) bloque la propagation directe

**Politique optimale :**
- La fleche `>` domine la ligne du haut (aller vers le but)
- En bas a gauche, on remonte (`^`) plutot que d'aller vers le piege
- La politique evite soigneusement le piege en contournant par le haut

## 5. Iteration de Politique

### Algorithme

1. Initialiser $\pi$ arbitrairement
2. Repeter jusqu'a stabilite :
   - **Evaluation** : Calculer $V^\pi$ (resoudre le systeme lineaire)
   - **Amelioration** : Mettre a jour $\pi(s) = \arg\max_a Q^\pi(s,a)$

Converge souvent **plus rapidement** que Value Iteration (moins d'iterations, mais chaque iteration est plus couteuse).

In [4]:
def policy_iteration(mdp: GridMDP, max_iter: int = 20):
    """Resolution d'un MDP par iteration de politique."""
    states = mdp.get_states()
    policy = {s: 'T' if s in mdp.terminal_states else 'N' for s in states}
    V = {s: 0.0 for s in states}
    
    for iteration in range(max_iter):
        # 1. Evaluation de politique (iterative simplifiee)
        for _ in range(50):
            new_V = dict(V)
            for s in states:
                if s in mdp.terminal_states:
                    new_V[s] = mdp.get_reward(s)
                    continue
                a = policy[s]
                v = mdp.get_reward(s)
                for next_s, prob in mdp.get_transitions(s, a):
                    v += mdp.gamma * prob * V[next_s]
                new_V[s] = v
            V = new_V
        
        # 2. Amelioration de politique
        stable = True
        for s in states:
            if s in mdp.terminal_states:
                continue
            old_action = policy[s]
            q_values = {}
            for a in mdp.ACTIONS:
                q = mdp.get_reward(s)
                for next_s, prob in mdp.get_transitions(s, a):
                    q += mdp.gamma * prob * V[next_s]
                q_values[a] = q
            policy[s] = max(q_values, key=q_values.get)
            if policy[s] != old_action:
                stable = False
        
        print(f"Iteration {iteration + 1} : stable = {stable}")
        if stable:
            break
    
    return V, policy


print("=== Iteration de Politique ===\n")
V_pi, policy_pi = policy_iteration(mdp)

print("\nComparaison avec Value Iteration :")
same = all(policy_vi[s] == policy_pi[s] for s in mdp.get_states())
if same:
    print("  Politiques identiques !")
else:
    for s in mdp.get_states():
        if policy_vi[s] != policy_pi[s]:
            print(f"  Difference en {s}: VI={policy_vi[s]}, PI={policy_pi[s]}")

=== Iteration de Politique ===

Iteration 1 : stable = False
Iteration 2 : stable = False
Iteration 3 : stable = True

Comparaison avec Value Iteration :
  Politiques identiques !


### Interpretation de l'Iteration de Politique

**Convergence rapide :**
- Seulement **3 iterations** contre 14 pour Value Iteration !
- Chaque iteration est plus couteuse (evaluation de politique), mais le nombre total est moindre

**Resultat identique :** Les deux methodes convergent vers la meme politique optimale $\pi^*$.

**Quand choisir quelle methode ?**
- Peu d'actions, grand espace d'etats : Policy Iteration
- Beaucoup d'actions, espace d'etats modere : Value Iteration

## 6. RTDP (Real-Time Dynamic Programming)

### Principe

RTDP est un algorithme de **planification en ligne** qui met a jour les valeurs uniquement pour les etats **atteignables** depuis un etat de depart.

**Avantage** : pas besoin d'explorer tout l'espace d'etats. Particulierement utile quand l'espace est grand mais seul un sous-ensemble est pertinent.

In [5]:
def rtdp(mdp: GridMDP, start_state: Tuple[int,int], n_trials: int = 100):
    """RTDP : planification en ligne par simulation."""
    states = mdp.get_states()
    V = {s: 0.0 for s in states}
    rng = np.random.RandomState(42)
    
    for trial in range(n_trials):
        s = start_state
        steps = 0
        
        while s not in mdp.terminal_states and steps < 100:
            # Mise a jour de Bellman
            q_values = {}
            for a in mdp.ACTIONS:
                q = mdp.get_reward(s)
                for next_s, prob in mdp.get_transitions(s, a):
                    q += mdp.gamma * prob * V[next_s]
                q_values[a] = q
            
            best_action = max(q_values, key=q_values.get)
            V[s] = q_values[best_action]
            
            # Simuler transition stochastique
            transitions = mdp.get_transitions(s, best_action)
            next_states = [t[0] for t in transitions]
            probs = [t[1] for t in transitions]
            idx = rng.choice(len(next_states), p=probs)
            s = next_states[idx]
            steps += 1
        
        if s in mdp.terminal_states:
            V[s] = mdp.get_reward(s)
    
    return V


print("=== RTDP (100 trials depuis (0,0)) ===\n")
V_rtdp = rtdp(mdp, (0, 0), n_trials=100)

print("Comparaison V_RTDP vs V_VI :")
print(f"  {'Etat':<8}| {'V_RTDP':>8} | {'V_VI':>8} | {'Diff':>8}")
print(f"  {'-'*8}|{'-'*10}|{'-'*10}|{'-'*9}")
for s in [(0,0), (0,2), (2,2), (3,2)]:
    diff = abs(V_rtdp[s] - V_vi[s])
    print(f"  ({s[0]},{s[1]})   | {V_rtdp[s]:8.4f} | {V_vi[s]:8.4f} | {diff:8.4f}")

=== RTDP (100 trials depuis (0,0)) ===

Comparaison V_RTDP vs V_VI :
  Etat    |   V_RTDP |     V_VI |     Diff
  --------|----------|----------|---------
  (0,0)   |   0.1666 |   0.2960 |   0.1294
  (0,2)   |  -0.1249 |   0.5094 |   0.6342
  (2,2)   |   0.7954 |   0.7954 |   0.0000
  (3,2)   |   1.0000 |   1.0000 |   0.0000


### Interpretation de RTDP

**Comparaison RTDP vs Value Iteration :**

| Etat | V_RTDP | V_VI | Ecart | Commentaire |
|------|--------|------|-------|-------------|
| (0,0) | faible | 0.296 | Grand | Peu visite par les trajectoires |
| (0,2) | faible | 0.509 | Grand | Loin du chemin optimal |
| (2,2) | ~0.795 | 0.795 | Faible | Sur le chemin optimal |
| (3,2) | 1.000 | 1.000 | Nul | Etat terminal |

**Conclusion** : RTDP est precis sur les etats atteignables depuis le start, mais imprécis sur les etats rarement visites. C'est un compromis : rapidite vs precision globale.

## 7. Reward Shaping

### Le probleme des recompenses sparses

Quand les recompenses sont rares (ex: +1 seulement au but), l'apprentissage est tres lent. Le **Reward Shaping** ajoute des recompenses intermediaires sans modifier la politique optimale.

### Theoreme (Ng et al., 1999)

Si $F(s,a,s') = \gamma \Phi(s') - \Phi(s)$ pour une fonction de potentiel $\Phi$, alors la politique optimale est **preservee**.

In [6]:
goal = (3, 2)

def phi(state, goal):
    """Fonction de potentiel : distance negative au but."""
    return -np.sqrt((state[0] - goal[0])**2 + (state[1] - goal[1])**2)


print("=== Reward Shaping avec Fonction de Potentiel ===\n")
print(f"But en {goal}")
print("Phi(s) = -distance(s, but)\n")

print("Fonction de potentiel Phi(s) :")
for y in range(2, -1, -1):
    row = ""
    for x in range(4):
        if (x, y) in mdp.walls:
            row += "  ####  "
        else:
            row += f" {phi((x,y), goal):6.2f} "
    print(row)

print("\nExemples de shaping reward F(s, a, s') = gamma*Phi(s') - Phi(s) :")
gamma = mdp.gamma
f01 = gamma * phi((1,0), goal) - phi((0,0), goal)
f10 = gamma * phi((0,0), goal) - phi((1,0), goal)
f23 = gamma * phi((3,2), goal) - phi((2,2), goal)
print(f"  F((0,0) -> (1,0)) = {f01:.3f} (vers le but)")
print(f"  F((1,0) -> (0,0)) = {f10:.3f} (loin du but)")
print(f"  F((2,2) -> (3,2)) = {f23:.3f} (atteindre le but)")
print()
print("=> Le shaping recompense les mouvements vers le but,")
print("   mais le theoreme garantit que la politique optimale est preservee.")

=== Reward Shaping avec Fonction de Potentiel ===

But en (3, 2)
Phi(s) = -distance(s, but)

Fonction de potentiel Phi(s) :
  -3.00   -2.00   -1.00   -0.00 
  -3.16   ####    -1.41   -1.00 
  -3.61   -2.83   -2.24   -2.00 

Exemples de shaping reward F(s, a, s') = gamma*Phi(s') - Phi(s) :
  F((0,0) -> (1,0)) = 1.060 (vers le but)
  F((1,0) -> (0,0)) = -0.417 (loin du but)
  F((2,2) -> (3,2)) = 1.000 (atteindre le but)

=> Le shaping recompense les mouvements vers le but,
   mais le theoreme garantit que la politique optimale est preservee.


---

### Synthese : Comparaison des methodes de resolution MDP

| Methode | Complexite/iter | Nb iterations | Avantage |
|---------|-----------------|---------------|----------|
| **Value Iteration** | $O(|S|^2 |A|)$ | ~14 | Simple, garantit convergence |
| **Policy Iteration** | $O(|S|^3 + |S|^2|A|)$ | ~3 | Tres rapide pour peu d'etats |
| **RTDP** | $O(\text{traj})$ | Variable | Echantillonne, pas besoin de tout explorer |

## 8. Bandits Multi-Bras

### Le probleme

Vous avez K machines a sous ("bras"). Chaque bras $i$ donne une recompense selon une distribution inconnue.

**Dilemme exploration-exploitation** :
- **Exploiter** : Tirer le bras qui semble meilleur
- **Explorer** : Tester d'autres bras pour mieux estimer leurs moyennes

In [7]:
class MultiArmedBandit:
    """Bandit multi-bras avec recompenses gaussiennes."""
    
    def __init__(self, means, seed=42):
        self.true_means = np.array(means)
        self.K = len(means)
        self.rng = np.random.RandomState(seed)
    
    def pull(self, arm):
        return self.true_means[arm] + 0.5 * self.rng.randn()
    
    @property
    def optimal_mean(self):
        return self.true_means.max()


class EpsilonGreedy:
    """Strategie epsilon-greedy."""
    
    def __init__(self, epsilon, seed=42):
        self.epsilon = epsilon
        self.name = f"epsilon-greedy (e={epsilon})"
        self.rng = np.random.RandomState(seed)
    
    def select_arm(self, counts, sum_rewards):
        if self.rng.rand() < self.epsilon:
            return self.rng.randint(len(counts))
        means = np.where(counts > 0, sum_rewards / np.maximum(counts, 1), 0)
        return np.argmax(means)


class UCB1:
    """Strategie Upper Confidence Bound."""
    
    @property
    def name(self):
        return "UCB1"
    
    def select_arm(self, counts, sum_rewards):
        total = counts.sum()
        if total < len(counts):
            return int(total)
        means = sum_rewards / counts
        bonus = np.sqrt(2 * np.log(total) / counts)
        return np.argmax(means + bonus)


def run_bandit(bandit, strategy, T=1000):
    counts = np.zeros(bandit.K, dtype=int)
    sum_rewards = np.zeros(bandit.K)
    total_reward = 0
    total_regret = 0
    
    for t in range(T):
        arm = strategy.select_arm(counts, sum_rewards)
        reward = bandit.pull(arm)
        counts[arm] += 1
        sum_rewards[arm] += reward
        total_reward += reward
        total_regret += bandit.optimal_mean - reward
    
    return total_reward, total_regret, counts


bandit = MultiArmedBandit([0.3, 0.5, 0.7, 0.4])
strategies = [EpsilonGreedy(0.1), UCB1()]

print(f"Bandit avec {bandit.K} bras, moyennes vraies inconnues")
print(f"Meilleure moyenne : {bandit.optimal_mean}\n")

for strategy in strategies:
    total_r, total_reg, counts = run_bandit(bandit, strategy)
    print(f"{strategy.name}:")
    print(f"  Recompense totale : {total_r:.1f}")
    print(f"  Regret cumule : {total_reg:.1f}")
    print(f"  Tirages par bras : {list(counts)}\n")

Bandit avec 4 bras, moyennes vraies inconnues
Meilleure moyenne : 0.7

epsilon-greedy (e=0.1):
  Recompense totale : 677.6
  Regret cumule : 22.4
  Tirages par bras : [np.int64(46), np.int64(22), np.int64(901), np.int64(31)]

UCB1:
  Recompense totale : 666.5
  Regret cumule : 33.5
  Tirages par bras : [np.int64(50), np.int64(174), np.int64(729), np.int64(47)]



### Interpretation des strategies de bandits

**Moyennes vraies (inconnues de l'agent) :** Bras 1=0.3, Bras 2=0.5, **Bras 3=0.7**, Bras 4=0.4

**Comparaison :**
- **$\varepsilon$-greedy** : avec $\varepsilon=0.1$, exploite fortement le meilleur bras identifie
- **UCB1** : explore plus systematiquement au debut, puis exploite

**Regret** : le regret cumule mesure la perte par rapport a l'oracle (qui connait le meilleur bras). Plus il est faible, meilleure est la strategie.

## 9. Indice de Gittins

### Le Theoreme Fondamental (Gittins, 1979)

L'indice de Gittins est l'un des resultats les plus elegants de la theorie des bandits : il transforme un probleme dynamique en une serie de decisions statiques.

**Resultat** : Pour un bandit stochastique avec recompenses escompteees, il existe un indice $\nu_i(t)$ pour chaque bras $i$ au temps $t$ tel que la politique optimale est simplement :

$$a_t = \arg\max_i \nu_i(t)$$

En pratique, le calcul exact est complexe. On utilise souvent **UCB1** comme approximation.

## 10. POMDPs : MDPs Partiellement Observables

### Motivation

Dans un MDP standard, l'agent connait l'etat exact du monde. En pratique, les capteurs sont **imparfaits** : l'agent n'observe qu'une partie de l'etat.

Un POMDP ajoute :
- Un ensemble d'**observations** $O$
- Une probabilite d'observation $P(o|s',a)$

L'agent maintient un **belief state** $b(s) = P(s|\text{historique})$ et met a jour ce belief via le **theoreme de Bayes**.

In [8]:
print("=== POMDP : Probleme du Tigre ===\n")
print("Deux portes : gauche (G) et droite (D)")
print("Un tigre est cache derriere l'une des portes.")
print("Actions : Ouvrir G, Ouvrir D, Ecouter\n")

p_correct = 0.85  # P(bruit_gauche | tigre_gauche)
reward_treasure = 10
reward_tiger = -100
cost_listen = -1

# Belief state : b = P(tigre_gauche)
b = 0.5

def eu_open_left(belief):
    return belief * reward_tiger + (1 - belief) * reward_treasure

def eu_open_right(belief):
    return belief * reward_treasure + (1 - belief) * reward_tiger

print(f"Belief initial : P(tigre gauche) = {b:.0%}\n")

print("Utilites esperees :")
print(f"  E[U(ouvrir gauche) | b={b:.0%}] = {eu_open_left(b):.1f}")
print(f"  E[U(ouvrir droite) | b={b:.0%}] = {eu_open_right(b):.1f}")
print(f"  Ecouter : cout immediat = {cost_listen}, mais reduit l'incertitude\n")

print("Simulation : 3 ecoutes donnent 'bruit gauche'\n")

for i in range(3):
    # Mise a jour bayesienne
    p_noise_left = b * p_correct + (1 - b) * (1 - p_correct)
    b = (p_correct * b) / p_noise_left
    print(f"Apres observation {i+1} : P(tigre gauche) = {b:.1%}")

print()
print(f"E[U(ouvrir gauche)] = {eu_open_left(b):.1f}")
print(f"E[U(ouvrir droite)] = {eu_open_right(b):.1f}")
print()
print(f"=> Decision : OUVRIR DROITE (tigre probablement a gauche)")

=== POMDP : Probleme du Tigre ===

Deux portes : gauche (G) et droite (D)
Un tigre est cache derriere l'une des portes.
Actions : Ouvrir G, Ouvrir D, Ecouter

Belief initial : P(tigre gauche) = 50%

Utilites esperees :
  E[U(ouvrir gauche) | b=50%] = -45.0
  E[U(ouvrir droite) | b=50%] = -45.0
  Ecouter : cout immediat = -1, mais reduit l'incertitude

Simulation : 3 ecoutes donnent 'bruit gauche'

Apres observation 1 : P(tigre gauche) = 85.0%
Apres observation 2 : P(tigre gauche) = 97.0%
Apres observation 3 : P(tigre gauche) = 99.5%

E[U(ouvrir gauche)] = -99.4
E[U(ouvrir droite)] = 9.4

=> Decision : OUVRIR DROITE (tigre probablement a gauche)


### Interpretation du probleme du tigre (POMDP)

**Evolution du belief state :**

| Etape | P(tigre gauche) | Decision optimale |
|-------|-----------------|-------------------|
| 0 | 50% | Ecouter (trop incertain) |
| 1 | 85% | Ecouter |
| 2 | 97% | Ecouter ou ouvrir droite |
| 3 | 99.5% | Ouvrir droite |

**Principe cle** : l'information a une valeur. Payer -1 pour ecouter est rationnel quand l'incertitude est elevee, car une mauvaise decision coute -100.

## 10bis. Belief State Updates : Maintenance Predictive

Le probleme du tigre calculait les mises a jour "a la main". Verifions le meme principe avec un **modele de maintenance predictive** plus complet, utilisant les operations matricielles numpy.

### Scenario
Un systeme industriel peut etre dans 3 etats : **Bon**, **Degrade**, **Defaillant**. Des capteurs de vibration fournissent des observations bruitees. On met a jour le belief state par prediction-correction (filtre bayesien).

In [9]:
# Matrice de transition (degradation naturelle)
trans_matrix = np.array([
    [0.90, 0.08, 0.02],   # Bon -> Bon, Degrade, Defaillant
    [0.00, 0.85, 0.15],   # Degrade -> Degrade, Defaillant
    [0.00, 0.00, 1.00],   # Defaillant (absorbant)
])

# Modele d'observation : P(obs | etat)
obs_matrix = np.array([
    [0.95, 0.05],   # Bon -> Normal, Anormal
    [0.30, 0.70],   # Degrade -> Normal, Anormal
    [0.05, 0.95],   # Defaillant -> Normal, Anormal
])

# Utilites des decisions
utilities = np.array([
    [ 100,   50, -500],   # Continuer
    [ -20,   80,  -50],   # Maintenance
    [-100, -100,   50],   # Remplacer
])
actions_nom = ["Continuer", "Maintenance", "Remplacer"]
etats_nom = ["Bon", "Degrade", "Defaillant"]

# Observations sequentielles : Normal, Anormal, Anormal
observations = [0, 1, 1]
obs_labels = ["Normal", "Anormal"]

# Belief initial (systeme venant d'etre installe)
belief = np.array([0.95, 0.04, 0.01])

print("=== Belief State Updates : Maintenance Predictive ===\n")
print("Belief initial :")
for e in range(3):
    print(f"  P({etats_nom[e]}) = {belief[e]:.1%}")
print()

for t in range(len(observations)):
    print(f"=== Pas de temps {t + 1} ===")
    
    # 1. Prediction : b_pred(s') = sum_s P(s'|s) * b(s)
    predicted = trans_matrix.T @ belief
    
    print("Apres prediction (transition) :")
    for e in range(3):
        print(f"  P({etats_nom[e]}) = {predicted[e]:.1%}")
    
    # 2. Correction bayesienne : P(s|obs) proportional a P(obs|s) * P(s)
    obs = observations[t]
    likelihood = obs_matrix[:, obs]
    unnormalized = likelihood * predicted
    belief = unnormalized / unnormalized.sum()
    
    print(f"Observation : {obs_labels[obs]}")
    print("Apres correction (Bayes) :")
    for e in range(3):
        bar = '#' * int(belief[e] * 30)
        print(f"  P({etats_nom[e]:10s}) = {belief[e]:.1%} {bar}")
    
    # 3. Decision basee sur l'utilite esperee
    EU = utilities @ belief
    best_action = np.argmax(EU)
    
    print("Utilites esperees :")
    for a in range(3):
        print(f"  E[U({actions_nom[a]:12s})] = {EU[a]:7.1f}")
    print(f"=> Decision : {actions_nom[best_action]}")
    print()

print("=== Resume ===")
print("Les observations 'Anormal' successives ont fait augmenter P(Degrade),")
print("declenchant le passage de 'Continuer' a 'Maintenance'.")
print("\nC'est le principe de la maintenance predictive bayesienne !")

=== Belief State Updates : Maintenance Predictive ===

Belief initial :
  P(Bon) = 95.0%
  P(Degrade) = 4.0%
  P(Defaillant) = 1.0%

=== Pas de temps 1 ===
Apres prediction (transition) :
  P(Bon) = 85.5%
  P(Degrade) = 11.0%
  P(Defaillant) = 3.5%
Observation : Normal
Apres correction (Bayes) :
  P(Bon       ) = 95.9% ############################
  P(Degrade   ) = 3.9% #
  P(Defaillant) = 0.2% 
Utilites esperees :
  E[U(Continuer   )] =    96.8
  E[U(Maintenance )] =   -16.2
  E[U(Remplacer   )] =   -99.7
=> Decision : Continuer

=== Pas de temps 2 ===
Apres prediction (transition) :
  P(Bon) = 86.3%
  P(Degrade) = 11.0%
  P(Defaillant) = 2.7%
Observation : Anormal
Apres correction (Bayes) :
  P(Bon       ) = 29.6% ########
  P(Degrade   ) = 52.7% ###############
  P(Defaillant) = 17.7% #####
Utilites esperees :
  E[U(Continuer   )] =   -32.3
  E[U(Maintenance )] =    27.4
  E[U(Remplacer   )] =   -73.5
=> Decision : Maintenance

=== Pas de temps 3 ===
Apres prediction (transition) :


### Interpretation de la maintenance predictive bayesienne

**Scenario simule** : Observations = [Normal, Anormal, Anormal]

**Evolution du belief et decisions :**

| Pas | Observation | P(Bon) | P(Degrade) | P(Defaillant) | Decision |
|-----|------------|--------|-----------|---------------|----------|
| 0 | - | 95% | 4% | 1% | - |
| 1 | Normal | ~96% | ~4% | ~0.2% | Continuer |
| 2 | Anormal | ~30% | ~53% | ~18% | Maintenance |
| 3 | Anormal | ~2% | ~56% | ~42% | Maintenance |

L'observation "Normal" renforce la certitude que le systeme est en bon etat. Les observations "Anormal" successives inversent le belief et declenchent une action de maintenance.

## 11. Lien avec la Serie RL

### Ce notebook : Concepts fondamentaux

- MDPs et equations de Bellman
- Methodes tabulaires (Value Iteration, Policy Iteration)
- Exploration vs exploitation (Bandits)
- Observabilite partielle (POMDPs)

### Serie RL : Extension a l'apprentissage

Quand $P(s'|s,a)$ et $R(s,a)$ sont inconnus, on utilise l'apprentissage par renforcement :
- **Q-Learning** : apprend $Q(s,a)$ par interaction
- **Deep RL** : approxime Q avec un reseau de neurones
- **Policy Gradient** : optimise directement la politique

---

## Exercice : MDP et Iteration de Valeur

**Objectifs** :
1. Implementer un MDP simple (grille 3x3)
2. Appliquer l'iteration de valeur pour trouver la politique optimale
3. Visualiser la fonction de valeur

**Indices** :
- Definir les etats comme des tuples (x, y) avec x,y in [0,2]
- Placer un but en (2,2) avec recompense +1
- Placer un obstacle en (1,1)
- Utiliser la classe GridMDP definie plus haut

In [10]:
# Exercice : MDP et Iteration de Valeur

# TODO etudiant : Creer un MDP grille 3x3
my_mdp = None  # TODO etudiant : utiliser GridMDP(3, 3, gamma=0.9)

# TODO etudiant : Definir but en (2,2) avec recompense +1

# TODO etudiant : Ajouter un obstacle en (1,1)

# TODO etudiant : Appliquer value_iteration() et afficher les resultats

print("Exercice a completer : implementez le MDP 3x3 et trouvez la politique optimale")

Exercice a completer : implementez le MDP 3x3 et trouvez la politique optimale


---

## 12. Resume

| Concept | Description |
|---------|-------------|
| **MDP** | $(S, A, P, R, \gamma)$ - cadre formel pour decisions sequentielles |
| **Bellman** | $V^*(s) = \max_a [R(s,a) + \gamma \sum P(s'|s,a) V^*(s')]$ |
| **Value Iteration** | Iteration sur les valeurs jusqu'a convergence |
| **Policy Iteration** | Alternance evaluation / amelioration de politique |
| **RTDP** | Planification en ligne, echantillonne les trajectoires |
| **Reward Shaping** | Guide l'apprentissage sans modifier la politique optimale |
| **Bandits** | Exploration-exploitation, $\varepsilon$-greedy, UCB1, Gittins |
| **POMDP** | MDP avec observations partielles, belief state bayesien |

### Navigation dans la serie

Ce notebook conclut la serie Decision Theory et fait le pont vers l'apprentissage par renforcement.

| Notebook | Thème |
|----------|-------|
| [PyMC-14](PyMC-14-Decision-Utility-Foundations.ipynb) | Fondements utilite |
| [PyMC-15](PyMC-15-Decision-Utility-Money.ipynb) | Utilite et argent |
| [PyMC-16](PyMC-16-Decision-Multi-Attribute.ipynb) | Multi-attributs |
| [PyMC-17](PyMC-17-Decision-Networks.ipynb) | Reseaux de decision |
| [PyMC-18](PyMC-18-Decision-Value-Information.ipynb) | Valeur de l'information |
| [PyMC-19](PyMC-19-Decision-Expert-Systems.ipynb) | Systemes experts |
| **PyMC-20** | **MDPs, Bandits, POMDPs** |